# Data Preprocessing

## Overview
This notebook handles the crucial data cleaning steps. Because agricultural data often contains outliers (e.g., misreported yields) and duplicate entries across districts, we apply stringent filtering rules.

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

## 1. Load Data
We load the raw inputs.

In [2]:
df = pd.read_csv('../data/processed/final_data_for_web.csv')
print(f'Initial Shape: {df.shape}')

Initial Shape: (249495, 14)


## 2. Invalid & Missing Data Removal
We discard rows with infinite or zero yields/areas. A yield of 0 or infinity is physically impossible or indicates failed collection.

In [3]:
df = df[np.isfinite(df['crop_yield']) & np.isfinite(df['area'])]
df = df[(df['crop_yield'] > 0) & (df['area'] > 0)]
df = df[df['crop_yield'] <= 500.0]  # Hard ceiling for extreme anomalies
print(f'Shape after validity checks: {df.shape}')

Shape after validity checks: (247360, 14)


## 3. Standardization
To prevent issues with duplicate representations, we convert string identifiers to lowercase and strip whitespace.

In [4]:
for col in ['season', 'crop', 'state_name', 'district_name']:
    df[col] = df[col].astype(str).str.lower().str.strip()
df.head()

,Unnamed: 0,state_name,district_name,crop_year,season,crop,area,temperature_2m_mean,precipitation_sum,relative_humidity_2m_mean,wind_speed_10m_mean,latitude,longitude,crop_yield
0,0,andhra pradesh,anantapur,2001,kharif,arhar/tur,27942.0,27.047742,1.125806,65.310753,21.16509,14.72422,77.430674,0.272994
1,1,andhra pradesh,anantapur,2001,kharif,bajra,1595.0,27.047742,1.125806,65.310753,21.16509,14.72422,77.430674,0.329781
2,2,andhra pradesh,anantapur,2001,kharif,castor seed,3191.0,27.047742,1.125806,65.310753,21.16509,14.72422,77.430674,0.282043
3,3,andhra pradesh,anantapur,2001,kharif,cotton(lint),8552.0,27.047742,1.125806,65.310753,21.16509,14.72422,77.430674,1.058817
4,4,andhra pradesh,anantapur,2001,kharif,dry chillies,3765.0,27.047742,1.125806,65.310753,21.16509,14.72422,77.430674,1.892961


## 4. Duplicate Handling
In government datasets, duplicate reports often occur. We enforce uniqueness on the primary geographical and temporal keys.

In [5]:
key_cols = ['state_name', 'district_name', 'season', 'crop', 'crop_year']
df = df.drop_duplicates(subset=key_cols, keep='first')
print(f'Shape after deduplication: {df.shape}')

Shape after deduplication: (247297, 14)


## 5. Statistical Outlier Capping (IQR)
To prevent the model from overfitting to wildly misreported yields, we calculate the IQR for each crop (in the training set only) and clip the yields. This is a critical defensive modeling strategy.

In [6]:
train_mask = df['crop_year'] <= 2017
for crop_name in df.loc[train_mask, 'crop'].unique():
    mask = train_mask & (df['crop'] == crop_name)
    vals = df.loc[mask, 'crop_yield']
    if len(vals) < 30: continue
    q1, q3 = vals.quantile(0.05), vals.quantile(0.95)
    iqr = q3 - q1
    lower, upper = max(0.001, q1 - 3.0 * iqr), q3 + 3.0 * iqr
    df.loc[mask, 'crop_yield'] = vals.clip(lower=lower, upper=upper)

print('Preprocessing complete.')

Preprocessing complete.
